# ICT CISD + SMT Sub-Spec Corpus — Transcript Extraction + Per-Video Concept Maps

**Scope (source-material ingestion only):** fetch transcripts for the two TTrades
confirmation-primitive videos in `cisd_smt_manifest.csv` (Change in State of Delivery, SMT
Divergence), normalize them to the structured timestamped schema, and produce **one concept
map PER VIDEO** of raw timestamped excerpts. **No interpretation, no summarization, no
definition synthesis, no reconciliation between the two videos.** Output is raw wording with
timestamps so a human (the ICT strategy chat) can build the CISD/SMT sub-spec downstream.

This is a **distinct mini-corpus** (`corpus=cisd_smt`). It stays **separate** from the 2022
mentorship corpus and the daily-bias corpus — it is never merged, cross-referenced, or folded
in.

The reusable, network-free parsing/tagging/windowing logic lives in
`quant_research.utils.transcript_ingest` and is unit-tested in
`tests/utils/test_transcript_ingest.py`. This notebook is just orchestration.

## Dependencies (approved + installed during Phase A — do NOT reinstall)

- `youtube-transcript-api` — primary fetch (manual captions, then its generated track).
- `yt-dlp` (CLI) — fallback for auto-subs via `--write-auto-subs --skip-download`.

`whisper` is intentionally **not** used. If BOTH backends fail for a video, the run STOPs for
that video, logs it to `results/cisd_smt_concept_map/_missing.csv`, and prints an operator
request to drop a transcript at
`data/raw/ict_sources/cisd_smt_transcripts/<video_id>.(json|txt|vtt)` and re-run.

**Kernel:** this notebook MUST run on the repo `.venv` kernel (`quant-venv`). A missing-backend
path (e.g. the anaconda base kernel) silently masquerades as "no transcript" — see lessons-log
2026-06-01. The setup cell asserts the expected backends are importable before fetching.

## Pipeline

1. **Fetch** — operator-supplied file (if present) first, else `youtube-transcript-api`, else
   `yt-dlp` fallback. Idempotent (skip videos whose normalized JSON already exists).
2. **Normalize** — one JSON per video at `data/processed/transcripts/cisd_smt/<video_id>.json`
   (LOCAL / gitignored — full third-party content).
3. **Concept map** — the `CISD_SMT_LEXICON` scanned against each video separately, emitting one
   per-video map with top-level `creator`/`role`/`video_id`/`corpus` metadata plus `±2`
   adjacent segments of context per hit. Maps are written to `docs/ict_bias_concept_maps/`
   (TRACKED) so the sub-spec resolves from a clean clone.

In [1]:
"""Setup: locate the repo root, wire imports, capture the git SHA, and define output paths."""

import shutil
import subprocess
import sys
from importlib import util as importlib_util
from pathlib import Path

# Locate repo root by walking up to the directory that holds pyproject.toml.
_here = Path.cwd().resolve()
REPO_ROOT = next(
    (p for p in (_here, *_here.parents) if (p / "pyproject.toml").is_file()),
    _here,
)
_src = REPO_ROOT / "src"
if _src.is_dir() and str(_src) not in sys.path:
    sys.path.insert(0, str(_src))

from quant_research.utils import transcript_ingest as ti  # noqa: E402

# CISD/SMT mini-corpus paths. RAW transcripts stay LOCAL (gitignored); concept MAPS are tracked.
MANIFEST_PATH = REPO_ROOT / "data" / "raw" / "ict_sources" / "cisd_smt_manifest.csv"
OPERATOR_DIR = REPO_ROOT / "data" / "raw" / "ict_sources" / "cisd_smt_transcripts"
TRANSCRIPT_DIR = REPO_ROOT / "data" / "processed" / "transcripts" / "cisd_smt"
MISSING_CSV = REPO_ROOT / "results" / "cisd_smt_concept_map" / "_missing.csv"
MAP_DIR = REPO_ROOT / "docs" / "ict_bias_concept_maps"

TRANSCRIPT_DIR.mkdir(parents=True, exist_ok=True)
MAP_DIR.mkdir(parents=True, exist_ok=True)


def _git_sha() -> str:
    """Return the short git SHA for artifact provenance, or 'unknown' if unavailable."""
    try:
        out = subprocess.run(
            ["git", "-C", str(REPO_ROOT), "rev-parse", "--short", "HEAD"],
            capture_output=True,
            text=True,
            check=True,
        )
        return out.stdout.strip() or "unknown"
    except Exception:
        return "unknown"


GIT_SHA = _git_sha()
HAS_YT_TRANSCRIPT_API = importlib_util.find_spec("youtube_transcript_api") is not None
HAS_YT_DLP = shutil.which("yt-dlp") is not None

# Guard against the lessons-log 2026-06-01 silent-failure footgun: at least one fetch backend
# must be importable on THIS kernel, otherwise every video would falsely log as 'no_transcript'.
assert HAS_YT_TRANSCRIPT_API or HAS_YT_DLP, (
    "No transcript backend on this kernel (youtube-transcript-api / yt-dlp). "
    "Run this notebook on the .venv 'quant-venv' kernel — see lessons-log 2026-06-01."
)

print(f"Repo root:          {REPO_ROOT}")
print(f"Manifest:           {MANIFEST_PATH}  (exists={MANIFEST_PATH.is_file()})")
print(f"Operator drop dir:  {OPERATOR_DIR}")
print(f"Transcript out dir: {TRANSCRIPT_DIR}  (LOCAL / gitignored)")
print(f"Concept map dir:    {MAP_DIR}  (TRACKED)")
print(f"git SHA:            {GIT_SHA}")
print(f"youtube-transcript-api installed: {HAS_YT_TRANSCRIPT_API}")
print(f"yt-dlp CLI available:             {HAS_YT_DLP}")

Repo root:          C:\Users\chris\Documents\Trading\quant-research-lab
Manifest:           C:\Users\chris\Documents\Trading\quant-research-lab\data\raw\ict_sources\cisd_smt_manifest.csv  (exists=True)
Operator drop dir:  C:\Users\chris\Documents\Trading\quant-research-lab\data\raw\ict_sources\cisd_smt_transcripts
Transcript out dir: C:\Users\chris\Documents\Trading\quant-research-lab\data\processed\transcripts\cisd_smt  (LOCAL / gitignored)
Concept map dir:    C:\Users\chris\Documents\Trading\quant-research-lab\docs\ict_bias_concept_maps  (TRACKED)
git SHA:            a2b041e
youtube-transcript-api installed: True
yt-dlp CLI available:             False


## Load the CISD/SMT manifest

Reads `data/raw/ict_sources/cisd_smt_manifest.csv` (columns:
`video_id, title, url, creator, role, corpus`) with the same `read_bias_manifest` reader. If
the file is missing this stops with an error — we do **not** guess video IDs. Video ids are
always handled as manifest VALUES (never bare CLI arguments).

In [2]:
rows = ti.read_bias_manifest(MANIFEST_PATH)  # raises FileNotFoundError if absent
assert all(r.corpus == "cisd_smt" for r in rows), "Unexpected corpus tag in manifest."
print(f"Loaded {len(rows)} CISD/SMT-manifest rows.\n")
for row in rows:
    print(
        f"  {row.video_id:<12}  {row.creator:<8} role={row.role:<8} corpus={row.corpus}  {row.title}"
    )

Loaded 2 CISD/SMT-manifest rows.

  CHIK5oBRKiw   ttrades  role=primary  corpus=cisd_smt  Change In State Of Delivery [CISD] - Orderblock Formation - ICT Concepts
  q1NmxUTm4n4   ttrades  role=primary  corpus=cisd_smt  SMT Divergence - ICT Concepts


## Steps 1 + 2 — Fetch and normalize (operator-supplied fallback)

For each manifest row: skip if its normalized JSON already exists (idempotent). Otherwise, if an
operator-supplied transcript exists at
`data/raw/ict_sources/cisd_smt_transcripts/<video_id>.(json|txt|vtt)`, parse it into the segment
schema (`source="operator_supplied"`). Otherwise fetch via `youtube-transcript-api`, then
`yt-dlp`. If BOTH network backends fail, the video is logged to
`results/cisd_smt_concept_map/_missing.csv` and a clear operator request is printed — no audio
transcription is attempted (whisper is not approved).

In [3]:
import csv
import json
import time

REQUEST_DELAY_S = 1.5  # polite pause between network fetches to avoid YouTube rate-limits


def _find_operator_file(video_id: str) -> Path | None:
    """Return an operator-supplied transcript file for ``video_id`` if one exists."""
    for ext in ("json", "vtt", "txt"):
        candidate = OPERATOR_DIR / f"{video_id}.{ext}"
        if candidate.is_file():
            return candidate
    return None


def _parse_operator_file(path: Path) -> list[dict]:
    """Parse an operator-supplied .json/.vtt/.txt file into raw segment dicts."""
    suffix = path.suffix.lower()
    if suffix == ".json":
        data = json.loads(path.read_text(encoding="utf-8"))
        if isinstance(data, dict) and "segments" in data:
            return list(data["segments"])
        return list(data)
    if suffix == ".vtt":
        return ti.parse_vtt(path.read_text(encoding="utf-8", errors="replace"))
    lines = [ln.strip() for ln in path.read_text(encoding="utf-8").splitlines() if ln.strip()]
    return [{"text": ln, "start": float(i), "end": float(i)} for i, ln in enumerate(lines)]


def _log_missing(row, reason: str) -> None:
    """Append a row to the CISD/SMT _missing.csv log, writing the header if the file is new."""
    MISSING_CSV.parent.mkdir(parents=True, exist_ok=True)
    write_header = not MISSING_CSV.exists()
    with MISSING_CSV.open("a", encoding="utf-8", newline="") as handle:
        writer = csv.writer(handle)
        if write_header:
            writer.writerow(ti.BIAS_MISSING_CSV_COLUMNS)
        writer.writerow([row.video_id, row.title, row.url, row.creator, reason])


fetched_count = skipped_count = missing_count = 0

for row in rows:
    out_path = TRANSCRIPT_DIR / f"{row.video_id}.json"
    if out_path.exists():
        skipped_count += 1
        print(f"  [skip]      {row.video_id}: normalized JSON already exists")
        continue

    operator_file = _find_operator_file(row.video_id)
    if operator_file is not None:
        source, raw_segments = "operator_supplied", _parse_operator_file(operator_file)
    else:
        time.sleep(REQUEST_DELAY_S)
        try:
            result = ti.fetch_transcript(row.video_id)  # video_id is a VALUE, never a CLI flag
        except Exception as exc:  # never abort the whole run on one failure
            result = None
            print(f"  [fetch-error] {row.video_id}: {exc!r}")
        if result is None:
            _log_missing(row, "no_transcript")
            missing_count += 1
            print(
                f"  [MISSING]   {row.video_id}: both backends failed.\n"
                f"    Transcript retrieval failed for {row.video_id}. Supply transcript at\n"
                f"    data/raw/ict_sources/cisd_smt_transcripts/{row.video_id}.(json|txt|vtt) "
                f"and re-run."
            )
            continue
        source, raw_segments = result

    document = ti.normalize_bias_transcript(
        video_id=row.video_id,
        title=row.title,
        url=row.url,
        creator=row.creator,
        role=row.role,
        corpus=row.corpus,
        source=source,
        raw_segments=raw_segments,
    )
    ti.write_json(out_path, document)
    fetched_count += 1
    print(f"  [{source:<16}] {row.video_id}: {len(document['segments'])} segments")

print(
    f"\nFetch summary: {fetched_count} written, {skipped_count} skipped (already present), "
    f"{missing_count} missing."
)
if missing_count:
    print(f"Missing videos logged to: {MISSING_CSV}")
    print("Supply the file(s) above and re-run this notebook to complete the corpus.")

  [auto_subs       ] CHIK5oBRKiw: 369 segments


  [auto_subs       ] q1NmxUTm4n4: 200 segments

Fetch summary: 2 written, 0 skipped (already present), 0 missing.


## Step 3 — Per-video concept maps (one per video, kept DISTINCT)

Load each normalized transcript and scan the **`CISD_SMT_LEXICON`** against it **separately**.
The two maps are never merged and conflicts are never reconciled — that is the downstream ICT
chat's job. Each map carries top-level `creator`/`role`/`video_id`/`corpus` metadata plus the
git SHA, and each hit includes `±2` adjacent segments of context. Pure keyword windowing — no
paraphrasing, no LLM summarization.

Output (TRACKED): `docs/ict_bias_concept_maps/ttrades_cisd.json` (from `CHIK5oBRKiw`) and
`docs/ict_bias_concept_maps/ttrades_smt.json` (from `q1NmxUTm4n4`).

In [4]:
MAP_FILENAMES = {
    "CHIK5oBRKiw": "ttrades_cisd.json",
    "q1NmxUTm4n4": "ttrades_smt.json",
}

written_maps = []
hit_tables: dict[str, dict[str, int]] = {}
for row in rows:
    transcript_path = TRANSCRIPT_DIR / f"{row.video_id}.json"
    if not transcript_path.exists():
        print(f"  [skip-map]  {row.video_id}: no normalized transcript yet (see _missing.csv)")
        continue
    with transcript_path.open("r", encoding="utf-8") as handle:
        transcript = json.load(handle)

    cmap = ti.build_bias_concept_map(transcript, lexicon=ti.CISD_SMT_LEXICON, git_sha=GIT_SHA)
    filename = MAP_FILENAMES.get(row.video_id, f"ttrades_{row.video_id}.json")
    out_path = ti.write_json(MAP_DIR / filename, cmap)
    written_maps.append(out_path)
    hit_tables[filename] = {c: len(h) for c, h in cmap["concepts"].items()}

# Per-group hit-count table for BOTH videos, side by side.
groups = list(ti.CISD_SMT_LEXICON)
files = list(hit_tables)
col_w = max((len(f) for f in files), default=10)
print(f"\n{'concept_group':<16}" + "".join(f"  {f:>{col_w}}" for f in files))
print("-" * (16 + sum(col_w + 2 for _ in files)))
for g in groups:
    print(f"{g:<16}" + "".join(f"  {hit_tables[f].get(g, 0):>{col_w}}" for f in files))
print("-" * (16 + sum(col_w + 2 for _ in files)))
print(f"{'total':<16}" + "".join(f"  {sum(hit_tables[f].values()):>{col_w}}" for f in files))

print("\nConcept maps written (TRACKED):")
for path in written_maps:
    print(f"  {path}")


concept_group     ttrades_cisd.json   ttrades_smt.json
------------------------------------------------------
cisd_core                        19                  0
cisd_mechanics                   14                  0
smt_core                          0                 24
smt_mechanics                     0                 12
instruments                       0                 14
shared_confirm                   23                  5
------------------------------------------------------
total                            56                 55

Concept maps written (TRACKED):
  C:\Users\chris\Documents\Trading\quant-research-lab\docs\ict_bias_concept_maps\ttrades_cisd.json
  C:\Users\chris\Documents\Trading\quant-research-lab\docs\ict_bias_concept_maps\ttrades_smt.json
